# BigMartSales Prediction with XGBoost
### Corrected Version

**Key Corrections Made:**
1. Fixed MLflow context for final model training
2. Improved feature engineering efficiency (vectorized operations)
3. Added proper error handling
4. Fixed data leakage in visibility ratio calculation
5. Added validation checks
6. Improved code organization

In [1]:
!pip install optuna mlflow xgboost -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207

## Imports and Configuration

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

import optuna
import mlflow
import xgboost as xgb
import mlflow.xgboost


# Configuration
RANDOM_STATE = 42
N_SPLITS = 5

mlflow.set_experiment("BigMartSales_Prediction")

2026/02/13 04:41:42 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/13 04:41:42 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/13 04:41:42 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/13 04:41:42 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/13 04:41:42 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/13 04:41:42 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/13 04:41:43 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/13 04:41:43 INFO mlflow.store.db.utils: Updating database tables
2026/02/13 04:41:43 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/13 04:41:43 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/02/13 04:41:43 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2026/02/13 04:4

<Experiment: artifact_location='/content/mlruns/1', creation_time=1770957704969, experiment_id='1', last_update_time=1770957704969, lifecycle_stage='active', name='BigMartSales_Prediction', tags={}>

## Data Loading

In [7]:
train = pd.read_csv("/content/train_v9rqX0R.csv")
test = pd.read_csv("/content/test_AbJTz2l.csv")

test_ids = test[["Item_Identifier", "Outlet_Identifier"]].copy()

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (8523, 12)
Test shape: (5681, 11)


## Feature Engineering Pipeline

**Improvements:**
- Vectorized operations instead of apply()
- Fixed data leakage in visibility ratio
- More efficient transformations

In [8]:
def feature_engineering(train, test):
    """
    Apply feature engineering transformations.

    Key fixes:
    - Vectorized operations for better performance
    - Proper handling of train/test statistics
    """
    train = train.copy()
    test = test.copy()

    # ===== Item Weight Imputation =====
    item_weight_median = train.groupby("Item_Type")["Item_Weight"].median()

    # Vectorized imputation (more efficient than apply)
    for df in [train, test]:
        missing_mask = df["Item_Weight"].isnull()
        df.loc[missing_mask, "Item_Weight"] = df.loc[missing_mask, "Item_Type"].map(item_weight_median)

    # ===== Item Visibility Handling =====
    train.loc[train["Item_Visibility"] == 0, "Item_Visibility"] = np.nan
    test.loc[test["Item_Visibility"] == 0, "Item_Visibility"] = np.nan

    visibility_median = train.groupby("Item_Type")["Item_Visibility"].median()

    for df in [train, test]:
        missing_mask = df["Item_Visibility"].isnull()
        df.loc[missing_mask, "Item_Visibility"] = df.loc[missing_mask, "Item_Type"].map(visibility_median)

    # ===== Fat Content Normalization =====
    mapping = {
        "lf": "Low Fat",
        "low fat": "Low Fat",
        "reg": "Regular",
        "regular": "Regular"
    }

    for df in [train, test]:
        df["Item_Fat_Content"] = (
            df["Item_Fat_Content"]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace(mapping)
        )

    # Binary encoding
    fat_encoding = {"Low Fat": 1, "Regular": 0}
    for df in [train, test]:
        df["Item_Fat_Content"] = df["Item_Fat_Content"].replace(fat_encoding)

    # ===== Outlet Age =====
    CURRENT_YEAR = 2013
    for df in [train, test]:
        df["Outlet_Age"] = CURRENT_YEAR - df["Outlet_Establishment_Year"]

    train.drop(columns=["Outlet_Establishment_Year"], inplace=True)
    test.drop(columns=["Outlet_Establishment_Year"], inplace=True)

    # ===== Visibility Ratio =====
    # FIXED: Calculate mean from train only to avoid data leakage
    visibility_mean_map = train.groupby("Item_Type")["Item_Visibility"].mean()

    train["Visibility_Ratio"] = train["Item_Visibility"] / train["Item_Type"].map(visibility_mean_map)
    test["Visibility_Ratio"] = test["Item_Visibility"] / test["Item_Type"].map(visibility_mean_map)

    # ===== MRP Binning =====
    bins = [0, 70, 140, 210, 300]
    train["MRP_Bin"] = pd.cut(train["Item_MRP"], bins=bins, labels=False)
    test["MRP_Bin"] = pd.cut(test["Item_MRP"], bins=bins, labels=False)

    # ===== Interaction Feature =====
    train["Item_Outlet_Type"] = train["Item_Type"] + "_" + train["Outlet_Type"]
    test["Item_Outlet_Type"] = test["Item_Type"] + "_" + test["Outlet_Type"]

    return train, test

## K-Fold Target Encoding

In [9]:
def kfold_target_encoding(train_df, test_df, column, target, n_splits=5):
    """
    K-fold target encoding to prevent overfitting.
    """
    train_encoded = pd.Series(index=train_df.index, dtype=float)
    global_mean = train_df[target].mean()

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    for train_idx, val_idx in kf.split(train_df):
        fold_train = train_df.iloc[train_idx]
        fold_val = train_df.iloc[val_idx]

        means = fold_train.groupby(column)[target].mean()
        train_encoded.iloc[val_idx] = fold_val[column].map(means)

    train_encoded.fillna(global_mean, inplace=True)

    # For test, use full train statistics
    full_means = train_df.groupby(column)[target].mean()
    test_encoded = test_df[column].map(full_means)
    test_encoded.fillna(global_mean, inplace=True)

    return train_encoded, test_encoded

## Build Final Dataset

In [10]:
def build_dataset(train, test):
    """
    Build the final modeling dataset.
    """
    train, test = feature_engineering(train, test)

    # Target encoding for high-cardinality features
    train["Item_TE"], test["Item_TE"] = kfold_target_encoding(
        train, test, "Item_Identifier", "Item_Outlet_Sales"
    )

    train["Outlet_TE"], test["Outlet_TE"] = kfold_target_encoding(
        train, test, "Outlet_Identifier", "Item_Outlet_Sales"
    )

    # Drop high-cardinality columns
    train.drop(columns=["Item_Identifier", "Outlet_Identifier"], inplace=True)
    test.drop(columns=["Item_Identifier", "Outlet_Identifier"], inplace=True)

    # One-hot encode remaining categoricals
    train = pd.get_dummies(train, drop_first=True)
    test = pd.get_dummies(test, drop_first=True)

    # Align train and test columns
    train, test = train.align(test, join="left", axis=1, fill_value=0)

    y = train["Item_Outlet_Sales"]
    X = train.drop("Item_Outlet_Sales", axis=1)

    return X, y, test

In [11]:
X, y, test_processed = build_dataset(train, test)

print(f"Train shape: {X.shape}")
print(f"Test shape: {test_processed.shape}")
print(f"\nFeature columns: {X.shape[1]}")

Train shape: (8523, 94)
Test shape: (5681, 95)

Feature columns: 94


## Model Evaluation Function

In [12]:
# def evaluate_xgb(params, X, y):
#     """
#     Evaluate XGBoost with cross-validation.

#     Returns:
#         cv_score: Mean RMSE across folds
#         avg_iter: Average best iteration
#     """
#     kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
#     fold_scores = []
#     best_iterations = []

#     for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
#         X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#         y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

#         dtrain = xgb.DMatrix(X_train, label=y_train)
#         dval = xgb.DMatrix(X_val, label=y_val)

#         xgb_params = {
#             "objective": "reg:squarederror",
#             "learning_rate": params["learning_rate"],
#             "max_depth": params["max_depth"],
#             "min_child_weight": params["min_child_weight"],
#             "subsample": params["subsample"],
#             "colsample_bytree": params["colsample_bytree"],
#             "alpha": params["reg_alpha"],
#             "lambda": params["reg_lambda"],
#             "seed": RANDOM_STATE,
#             "eval_metric": "rmse"
#         }

#         model = xgb.train(
#             xgb_params,
#             dtrain,
#             num_boost_round=5000,
#             evals=[(dval, "validation")],
#             early_stopping_rounds=200,
#             verbose_eval=False
#         )

#         preds = model.predict(dval)
#         rmse = np.sqrt(mean_squared_error(y_val, preds))

#         fold_scores.append(rmse)
#         best_iterations.append(model.best_iteration)

#         print(f"Fold {fold+1} RMSE: {rmse:.4f}")

#     cv_score = np.mean(fold_scores)
#     avg_iter = int(np.mean(best_iterations))

#     print(f"\nCV RMSE: {cv_score:.4f}")
#     print(f"Avg Best Iter: {avg_iter}")

#     return cv_score, avg_iter

In [21]:
def evaluate_xgb_adaptive(params, X, y, verbose=True):
    """
    Evaluate XGBoost with ADAPTIVE iteration count based on learning rate
    """
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_scores = []
    best_iterations = []

    # Adaptive max iterations based on learning rate
    lr = params["learning_rate"]
    if lr < 0.01:
        max_iter = 20000
    elif lr < 0.03:
        max_iter = 15000
    elif lr < 0.05:
        max_iter = 10000
    else:
        max_iter = 5000

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        dtrain = xgb.DMatrix(X_train, label=y_train)
        dval = xgb.DMatrix(X_val, label=y_val)

        xgb_params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "tree_method": "hist",
            "seed": RANDOM_STATE,
            **params
        }

        model = xgb.train(
            xgb_params,
            dtrain,
            num_boost_round=max_iter,
            evals=[(dval, "validation")],
            early_stopping_rounds=200,
            verbose_eval=False
        )

        preds = model.predict(dval)
        rmse = np.sqrt(mean_squared_error(y_val, preds))

        fold_scores.append(rmse)
        best_iterations.append(model.best_iteration)

        if verbose:
            print(f"  Fold {fold+1} RMSE: {rmse:.4f} (iters: {model.best_iteration})")

    cv_score = np.mean(fold_scores)
    avg_iter = int(np.mean(best_iterations))

    if verbose:
        print(f"  → CV RMSE: {cv_score:.4f} (avg iters: {avg_iter})")

    return cv_score, avg_iter

## Baseline Model

In [13]:
# xgb_params = {
#     "learning_rate": 0.01,
#     "max_depth": 6,
#     "min_child_weight": 3,
#     "subsample": 0.8,
#     "colsample_bytree": 0.8,
#     "reg_alpha": 0.1,
#     "reg_lambda": 1
# }

In [22]:
print("="*60)
print("STEP 1: Testing XGBoost with Good Defaults")
print("="*60)

good_defaults = {
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "gamma": 0.1
}

with mlflow.start_run(run_name="Good_Baseline"):
    baseline_cv, baseline_iter = evaluate_xgb_adaptive(good_defaults, X, y)

    mlflow.log_params(good_defaults)
    mlflow.log_metric("cv_rmse", baseline_cv)
    mlflow.log_metric("best_iteration", baseline_iter)

    print(f"\n✓ Baseline RMSE: {baseline_cv:.4f}")
    print(f"  (Your Optuna result: 1098.66)")
    if baseline_cv < 1098.66:
        print(f"  → Improvement: {1098.66 - baseline_cv:.2f} RMSE")


STEP 1: Testing XGBoost with Good Defaults
  Fold 1 RMSE: 1034.1591 (iters: 77)
  Fold 2 RMSE: 1088.1234 (iters: 65)
  Fold 3 RMSE: 1100.2120 (iters: 63)
  Fold 4 RMSE: 1124.2213 (iters: 95)
  Fold 5 RMSE: 1231.5342 (iters: 53)
  → CV RMSE: 1115.6500 (avg iters: 70)

✓ Baseline RMSE: 1115.6500
  (Your Optuna result: 1098.66)


In [14]:
# with mlflow.start_run(run_name="XGB_Baseline"):
#     baseline_cv, baseline_iter = evaluate_xgb(xgb_params, X, y)

#     mlflow.log_params(xgb_params)
#     mlflow.log_metric("cv_rmse", baseline_cv)
#     mlflow.log_metric("best_iteration", baseline_iter)

#     print("\nBaseline logged to MLflow.")

Fold 1 RMSE: 1019.2106
Fold 2 RMSE: 1079.5442
Fold 3 RMSE: 1079.5863
Fold 4 RMSE: 1118.6649
Fold 5 RMSE: 1228.8152

CV RMSE: 1105.1642
Avg Best Iter: 397

Baseline logged to MLflow.


In [23]:
print("\n" + "="*60)
print("STEP 2a: Stage 1 - Finding Learning Rate & Depth")
print("="*60)

def objective_stage1(trial):
    with mlflow.start_run(nested=True):
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
            "max_depth": trial.suggest_int("max_depth", 5, 10),
            "min_child_weight": 3,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0,
            "gamma": 0.1
        }

        print(f"\nTrial {trial.number + 1}: lr={params['learning_rate']:.4f}, depth={params['max_depth']}")
        cv_score, avg_iter = evaluate_xgb_adaptive(params, X, y, verbose=True)

        mlflow.log_params(params)
        mlflow.log_metric("cv_rmse", cv_score)
        mlflow.log_metric("avg_best_iteration", avg_iter)

        return cv_score

with mlflow.start_run(run_name="Stage1_Optimization"):
    study_stage1 = optuna.create_study(direction="minimize")
    study_stage1.optimize(objective_stage1, n_trials=25, show_progress_bar=True)

    mlflow.log_params(study_stage1.best_params)
    mlflow.log_metric("best_cv_rmse", study_stage1.best_value)

    print(f"\n{'='*60}")
    print(f"Stage 1 Complete!")
    print(f"Best CV RMSE: {study_stage1.best_value:.4f}")
    print(f"Best LR: {study_stage1.best_params['learning_rate']:.4f}")
    print(f"Best Depth: {study_stage1.best_params['max_depth']}")
    print(f"{'='*60}")


[I 2026-02-13 05:00:14,063] A new study created in memory with name: no-name-a0a66716-aab9-449f-aa81-428a0fa820e3



STEP 2a: Stage 1 - Finding Learning Rate & Depth


  0%|          | 0/25 [00:00<?, ?it/s]


Trial 1: lr=0.0496, depth=6
  Fold 1 RMSE: 1031.2680 (iters: 77)
  Fold 2 RMSE: 1090.5955 (iters: 64)
  Fold 3 RMSE: 1098.4060 (iters: 77)
  Fold 4 RMSE: 1124.9484 (iters: 95)
  Fold 5 RMSE: 1237.8479 (iters: 53)
  → CV RMSE: 1116.6132 (avg iters: 73)
[I 2026-02-13 05:00:19,105] Trial 0 finished with value: 1116.613174308985 and parameters: {'learning_rate': 0.04963645244688345, 'max_depth': 6}. Best is trial 0 with value: 1116.613174308985.

Trial 2: lr=0.0506, depth=9
  Fold 1 RMSE: 1054.8817 (iters: 64)
  Fold 2 RMSE: 1106.7685 (iters: 56)
  Fold 3 RMSE: 1124.4482 (iters: 47)
  Fold 4 RMSE: 1139.1809 (iters: 86)
  Fold 5 RMSE: 1243.8131 (iters: 51)
  → CV RMSE: 1133.8185 (avg iters: 60)
[I 2026-02-13 05:00:29,453] Trial 1 finished with value: 1133.8184897072465 and parameters: {'learning_rate': 0.050569014897158475, 'max_depth': 9}. Best is trial 0 with value: 1116.613174308985.

Trial 3: lr=0.0283, depth=7
  Fold 1 RMSE: 1030.9819 (iters: 97)
  Fold 2 RMSE: 1087.4987 (iters: 105)


In [24]:
print("\n" + "="*60)
print("STEP 2b: Stage 2 - Fine-tuning Regularization")
print("="*60)

def objective_stage2(trial):
    with mlflow.start_run(nested=True):
        params = {
            "learning_rate": study_stage1.best_params["learning_rate"],  # FIXED from stage 1
            "max_depth": study_stage1.best_params["max_depth"],  # FIXED from stage 1
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 7),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 0, 2.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 3.0),
            "gamma": trial.suggest_float("gamma", 0, 1.5),
        }

        print(f"\nTrial {trial.number + 1}")
        cv_score, avg_iter = evaluate_xgb_adaptive(params, X, y, verbose=True)

        mlflow.log_params(params)
        mlflow.log_metric("cv_rmse", cv_score)
        mlflow.log_metric("avg_best_iteration", avg_iter)

        return cv_score

with mlflow.start_run(run_name="Stage2_Optimization"):
    study_stage2 = optuna.create_study(direction="minimize")
    study_stage2.optimize(objective_stage2, n_trials=35, show_progress_bar=True)

    mlflow.log_params(study_stage2.best_params)
    mlflow.log_metric("best_cv_rmse", study_stage2.best_value)

    print(f"\n{'='*60}")
    print(f"Stage 2 Complete!")
    print(f"Best CV RMSE: {study_stage2.best_value:.4f}")
    print(f"{'='*60}")

[I 2026-02-13 05:04:56,802] A new study created in memory with name: no-name-45635ab1-a687-46cd-b575-0a0827982416



STEP 2b: Stage 2 - Fine-tuning Regularization


  0%|          | 0/35 [00:00<?, ?it/s]


Trial 1
  Fold 1 RMSE: 1018.7630 (iters: 189)
  Fold 2 RMSE: 1079.6189 (iters: 243)
  Fold 3 RMSE: 1079.8724 (iters: 163)
  Fold 4 RMSE: 1115.0573 (iters: 269)
  Fold 5 RMSE: 1228.9959 (iters: 119)
  → CV RMSE: 1104.4615 (avg iters: 196)
[I 2026-02-13 05:05:04,845] Trial 0 finished with value: 1104.461501072144 and parameters: {'min_child_weight': 1, 'subsample': 0.6702541807260143, 'colsample_bytree': 0.8406173440900159, 'reg_alpha': 1.5647269848081167, 'reg_lambda': 1.6186144755329226, 'gamma': 0.07149695458618732}. Best is trial 0 with value: 1104.461501072144.

Trial 2
  Fold 1 RMSE: 1021.6756 (iters: 184)
  Fold 2 RMSE: 1079.6862 (iters: 158)
  Fold 3 RMSE: 1078.0204 (iters: 198)
  Fold 4 RMSE: 1116.2448 (iters: 383)
  Fold 5 RMSE: 1259.2843 (iters: 109)
  → CV RMSE: 1110.9823 (avg iters: 206)
[I 2026-02-13 05:05:11,197] Trial 1 finished with value: 1110.9822631601314 and parameters: {'min_child_weight': 2, 'subsample': 0.9321011698775834, 'colsample_bytree': 0.9306238897251007, 

In [28]:
print("\n" + "="*60)
print("STEP 3: Training Final Model")
print("="*60)

# Combine best parameters from Stage 1 and Stage 2
best_params = {
    "learning_rate": study_stage1.best_params["learning_rate"],
    "max_depth": study_stage1.best_params["max_depth"],
    **study_stage2.best_params
}

with mlflow.start_run(run_name="Final_XGB_Model"):
    # Determine optimal n_estimators
    lr = best_params["learning_rate"]
    if lr < 0.01:
        n_estimators = 20000
    elif lr < 0.03:
        n_estimators = 15000
    elif lr < 0.05:
        n_estimators = 10000
    else:
        n_estimators = 5000

    final_model = xgb.XGBRegressor(
        n_estimators=n_estimators,
        **best_params,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        early_stopping_rounds=200
    )

    print(f"Training with {n_estimators} max iterations...")
    final_model.fit(
        X, y,
        eval_set=[(X, y)],
        verbose=False
    )

    mlflow.log_params(best_params)
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("actual_iterations", final_model.best_iteration)
    mlflow.xgboost.log_model(final_model, "final_xgb_model")

    print(f"✓ Model trained in {final_model.best_iteration} iterations")


STEP 3: Training Final Model
Training with 15000 max iterations...


2026/02/13 05:13:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✓ Model trained in 14999 iterations


In [30]:
# Align test features to match training
test_processed = test_processed.reindex(columns=X.columns, fill_value=0)

# Now predict
predictions = final_model.predict(test_processed)

In [32]:
predictions = final_model.predict(test_processed)

submission = test_ids.copy()
submission["Item_Outlet_Sales"] = predictions

submission.to_csv("submission_5.csv", index=False)

print(f"\n{'='*60}")
print("RESULTS SUMMARY")
print(f"{'='*60}")
print(f"Original Optuna RMSE:  1098.66")
print(f"Good Baseline RMSE:    {baseline_cv:.4f}")
print(f"Stage 1 Best RMSE:     {study_stage1.best_value:.4f}")
print(f"Stage 2 Best RMSE:     {study_stage2.best_value:.4f}")
print(f"\nImprovement: {1098.66 - study_stage2.best_value:.2f} RMSE")
print(f"\nSubmission saved: submission_5.csv")
print(f"{'='*60}")


RESULTS SUMMARY
Original Optuna RMSE:  1098.66
Good Baseline RMSE:    1115.6500
Stage 1 Best RMSE:     1104.3980
Stage 2 Best RMSE:     1100.4759

Improvement: -1.82 RMSE

Submission saved: submission_5.csv
